# Salary Prediction Model - Optimized Training Pipeline

This notebook engineers consistent HR features, compares several CPU-friendly regressors on a validation split, selects the lowest-MAE candidate, evaluates once on a held-out test set, and saves the complete inference pipeline. It expects `data/salary_data_v2.xlsx` with a `Salary (INR)` target and the profile columns documented below.

**Important:** the model artifact is created only after this notebook is executed with the actual dataset. The app does not train at runtime.


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from scripts.salary_features import (
    RAW_FEATURES, MODEL_FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
    add_engineered_features, city_to_tier,
)

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
DATA_PATH = Path("data/salary_data_v2.xlsx")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}. Place salary_data_v2.xlsx in the data/ folder.")


## 1. Load and clean data

Identifiers and joining dates are excluded to avoid leakage. Invalid or non-positive salary targets are removed; experience is numeric, non-negative and imputed. City tier is derived consistently from city names when a City column is available.


In [2]:
df = pd.read_excel(DATA_PATH)

if "Salary (INR)" not in df.columns and "Annual Salary (INR)" in df.columns:
    df = df.rename(columns={"Annual Salary (INR)": "Salary (INR)"})

df = df.drop(columns=["Employee ID", "Email", "Random Code", "Joining Date"], errors="ignore")

if "Salary (INR)" not in df.columns:
    raise KeyError("Expected target column 'Salary (INR)' or 'Annual Salary (INR)' was not found in the spreadsheet.")
if "Experience (Years)" not in df.columns:
    raise KeyError("Expected feature column 'Experience (Years)' was not found in the spreadsheet.")

if "City" in df.columns:
    df["City Tier"] = df["City"].map(city_to_tier)
elif "City Tier" not in df.columns:
    raise KeyError("Provide either a 'City' column or a precomputed 'City Tier' column.")

missing_features = [col for col in RAW_FEATURES if col not in df.columns]
if missing_features:
    raise KeyError(f"Required input columns missing from dataset: {missing_features}")

df["Salary (INR)"] = pd.to_numeric(df["Salary (INR)"], errors="coerce")
df["Experience (Years)"] = pd.to_numeric(df["Experience (Years)"], errors="coerce")
df = df.dropna(subset=["Salary (INR)"])
df = df[df["Salary (INR)"] > 0].copy()
if df.empty:
    raise ValueError("No rows with a positive numeric salary target remain after cleaning.")
exp_median = df["Experience (Years)"].clip(lower=0).median()
df["Experience (Years)"] = df["Experience (Years)"].fillna(exp_median if pd.notna(exp_median) else 0).clip(lower=0)

X = add_engineered_features(df[RAW_FEATURES])
y = df["Salary (INR)"].astype(float)
print(f"Rows ready: {len(df):,} | engineered features: {X.shape[1]}")
print(f"Target salary median: ₹{y.median():,.0f} | 5th–95th percentile: ₹{y.quantile(.05):,.0f}–₹{y.quantile(.95):,.0f}")


Rows ready: 11,000 | engineered features: 13
Target salary median: ₹2,152,150 | 5th–95th percentile: ₹908,025–₹4,678,032


## 2. Preprocessing and model candidates

The target is modelled on a log scale and converted back to INR by `TransformedTargetRegressor`; this is often more stable for right-skewed salary distributions. Rare one-hot categories are grouped with `min_frequency=5`, unseen inference categories are ignored safely, and numeric/categorical imputers keep the pipeline robust. Tree models do not require feature scaling.


In [3]:
def make_estimator(model_name: str):
    if model_name == "Extra Trees":
        regressor = ExtraTreesRegressor(
            n_estimators=250, max_features=0.85, min_samples_leaf=2,
            n_jobs=-1, random_state=RANDOM_STATE,
        )
    elif model_name == "Random Forest":
        regressor = RandomForestRegressor(
            n_estimators=200, max_features=0.80, min_samples_leaf=2,
            n_jobs=-1, random_state=RANDOM_STATE,
        )
    elif model_name == "Histogram Gradient Boosting":
        regressor = HistGradientBoostingRegressor(
            max_iter=250, learning_rate=0.06, max_leaf_nodes=31,
            l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=20, random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(f"Unknown estimator: {model_name}")

    numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False, dtype=np.float32)),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ], remainder="drop", verbose_feature_names_out=False)
    pipe = Pipeline([("preprocess", preprocessor), ("regressor", regressor)])
    return TransformedTargetRegressor(regressor=pipe, func=np.log1p, inverse_func=np.expm1, check_inverse=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE,
)
X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train, y_train, test_size=0.20, random_state=RANDOM_STATE,
)

candidate_names = ["Extra Trees", "Random Forest", "Histogram Gradient Boosting"]
validation_rows = []
for name in candidate_names:
    candidate = make_estimator(name)
    candidate.fit(X_fit, y_fit)
    val_pred = np.maximum(0, candidate.predict(X_valid))
    validation_rows.append({
        "Model": name,
        "Validation MAE (₹)": mean_absolute_error(y_valid, val_pred),
        "Validation RMSE (₹)": np.sqrt(mean_squared_error(y_valid, val_pred)),
        "Validation R²": r2_score(y_valid, val_pred),
    })

validation_results = pd.DataFrame(validation_rows).sort_values("Validation MAE (₹)").reset_index(drop=True)
display(validation_results)
best_name = validation_results.loc[0, "Model"]
print(f"Selected on validation MAE: {best_name}")


,Model,Validation MAE (₹),Validation RMSE (₹),Validation R²
0,Histogram Gradient Boosting,198391.585872,281855.923051,0.945574
1,Extra Trees,285617.918243,401874.180554,0.889356
2,Random Forest,290712.827393,416163.717450,0.881347


Selected on validation MAE: Histogram Gradient Boosting


## 3. Refit selected engine and evaluate on held-out test data

The test set is held aside until candidate selection is complete. The chosen pipeline is then refit using all training rows and evaluated once on the untouched test set. MAE and RMSE are in INR; MAPE is reported as a secondary relative-error metric.


In [4]:
best_model = make_estimator(str(best_name))
best_model.fit(X_train, y_train)
test_pred = np.maximum(0, best_model.predict(X_test))
metrics = {
    "selected_model": best_name,
    "test_rows": int(len(y_test)),
    "MAE_INR": float(mean_absolute_error(y_test, test_pred)),
    "RMSE_INR": float(np.sqrt(mean_squared_error(y_test, test_pred))),
    "R2": float(r2_score(y_test, test_pred)),
    "MAPE_percent": float(np.mean(np.abs((y_test.to_numpy() - test_pred) / y_test.to_numpy())) * 100),
    "feature_count": int(X.shape[1]),
    "training_rows": int(len(X_train)),
}
print(json.dumps(metrics, indent=2))


{
  "selected_model": "Histogram Gradient Boosting",
  "test_rows": 2200,
  "MAE_INR": 197355.28107265453,
  "RMSE_INR": 275364.274136476,
  "R2": 0.9464102808059474,
  "MAPE_percent": 8.31567412712697,
  "feature_count": 13,
  "training_rows": 8800
}


## 4. Save deployable artifact and model card

The complete preprocessing + regression pipeline is serialized together, so inference uses the same encoding rules as training. Compression reduces artifact size. The Streamlit app caches the artifact in memory and batches scenario predictions to avoid retraining or repeated single-row calls at runtime.


In [5]:
model_path = MODEL_DIR / "salary_model.pkl"
metadata_path = MODEL_DIR / "model_metadata.json"
joblib.dump(best_model, model_path, compress=3)
metadata = {
    **metrics,
    "raw_features": RAW_FEATURES,
    "engineered_features": MODEL_FEATURES,
    "target": "Salary (INR)",
    "target_transform": "log1p / expm1",
    "rare_category_min_frequency": 5,
    "validation_candidates": validation_results.to_dict(orient="records"),
    "notes": "Estimate is decision support, not a validated salary confidence interval. Evaluate fairness and dataset coverage before operational HR use.",
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved model: {model_path.resolve()} ({model_path.stat().st_size / (1024**2):.2f} MB)")
print(f"Saved metadata: {metadata_path.resolve()}")


Saved model: C:\Users\goure\OneDrive\Documents\python course 2025\my_projects\Salary_prediction_project\models\salary_model.pkl (0.35 MB)
Saved metadata: C:\Users\goure\OneDrive\Documents\python course 2025\my_projects\Salary_prediction_project\models\model_metadata.json


## Operational notes

- Keep `salary_features.py`, `app.py`, and `models/salary_model.pkl` together in the project root/layout expected by the app.
- Do not commit private employee-level data or personally identifiable information to a public repository.
- Re-run training when the dataset changes; do not train in the Streamlit request path.
- Before using estimates for compensation decisions, assess sample size by role/location, temporal coverage, systematic error across groups, and the consequences of under/over-estimation.
